In [ ]:
# Import all required libraries, set up required functions, and set directory paths

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Function for plotting predicted values against actual values for each lattice parameter (a, b, and c)
# def PlotPredictions(Y_test, Y_pred_df, modelName, displayStatistics=True, MSEVals=None, MAEVals=None, R2Vals=None, df=None, col=None, labels=None, displayOthers=True):
#     fig, axes = plt.subplots(1, 3, figsize=(18, 10))

#     Y_test_df = Y_test.reset_index(drop=True)

#     # Creates 3 subplots for each lattice parameter
#     for i, param in enumerate(Y_test_df.columns):
#         ax = axes[i]

#         #This plots each point colored based off its chosen label if X_test_full and labels is passed
#         if col is not None and labels is not None:            
#             X_test_full = df.loc[Y_test.index].reset_index(drop=True)
            
#             arr = np.asarray(X_test_full[col])
#             missing = [f for f in labels if f not in arr]
#             if missing:
#                 raise ValueError(f"The following labels are not found in column {col} of X_test_full: {missing}")

#             labelsList = list(labels)
#             cmap = plt.get_cmap('tab10')
#             colors = {label: cmap(i % cmap.N) for i, label in enumerate(labelsList)}
            
#             labelsList = ["Other"] + labelsList
#             colors["Other"] = "gray"
            
#             for label in labelsList:
#                 if label == 'Other':
#                     mask = ~np.isin(X_test_full[col], labels)
#                     if not displayOthers or not mask.any():
#                         continue
#                 else:      
#                     mask = X_test_full[col] == label
                    
#                 # Plot each actual value against its predicted value as individual points
#                 ax.scatter(
#                     Y_test_df.loc[mask, param],
#                     Y_pred_df.loc[mask, f"{param}_pred"],
#                     color = colors[label],
#                     label = label.title() if mask is not None else None,
#                     alpha = 0.7
#                 )

#                 # Create line of perfect fit where actual value = predicted value
#                 line = [Y_test_df[param].min(), Y_test_df[param].max()]
#                 ax.plot(line, line, 'k--')
            
#             if col == 'crystal_system':
#                 ax.legend(title='Crystal System', fontsize=18, title_fontsize=18)
#             else:
#                 ax.legend(title=col, fontsize=18, title_fontsize=18)
        
#         #This plots each point as the same color
#         else:
#             # Plot each actual value against its predicted value as individual points
#             ax.scatter(Y_test_df[param], Y_pred_df[f"{param}_pred"])

#             # Create line of perfect fit where actual value = predicted value
#             line = [Y_test_df[param].min(), Y_test_df[param].max()]
#             ax.plot(line, line, 'r--')

#         ax.set_xlabel("Actual Value", fontsize=18)
#         ax.tick_params(labelsize=18)
#         ax.set_ylabel("Predicted Value", fontsize=18)
#         ax.set_title(f"Lattice Parameter '{param}' (Å)", fontsize=20)
        
#         # Display MSE and R2 statistics for each lattice parameter if desired
#         if displayStatistics:
#             if MSEVals is not None and MAEVals is not None and R2Vals is not None:
#                 if df is not None:
#                     text = f"MSE={MSEVals[i]:.4f}\nMAE={MAEVals[i]:.4f}\nR²={R2Vals[i]:.4f}\nTest Dataset Size={len(df)}"
#                 else:
#                     text = f"MSE={MSEVals[i]:.4f}\nMAE={MAEVals[i]:.4f}\nR²={R2Vals[i]:.4f}"
                
#                 ax.text(
#                     0.05, 0.95,
#                     text,
#                     transform=ax.transAxes,
#                     verticalalignment='top',
#                     bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor="gray"),
#                     fontsize=14
#                 )
    
#     fig.suptitle(f"Predicted vs. Actual Lattice Parameter Using {modelName}", fontsize=22)
    
#     plt.tight_layout()
#     plt.show()    
def PlotPredictions(
    Y_test, Y_pred_df, modelName,
    displayStatistics=True, MSEVals=None, MAEVals=None, R2Vals=None,
    df=None, col=None, labels=None, displayOthers=True
):
    fig, axes = plt.subplots(1, 3, figsize=(18, 10))
    Y_test_df = Y_test.reset_index(drop=True)

    for i, param in enumerate(Y_test_df.columns):
        ax = axes[i]

        if col is not None and labels is not None:
            X_test_full = df.loc[Y_test.index].reset_index(drop=True)
            arr = np.asarray(X_test_full[col])
            missing = [f for f in labels if f not in arr]
            if missing:
                raise ValueError(f"The following labels are not found in column {col} of X_test_full: {missing}")

            labelsList = list(labels)
            cmap = plt.get_cmap('tab10')
            colors = {label: cmap(idx % cmap.N) for idx, label in enumerate(labelsList)}
            labelsList = ["Other"] + labelsList
            colors["Other"] = "gray"

            for label in labelsList:
                if label == "Other":
                    mask = ~np.isin(X_test_full[col], labels)
                    if not displayOthers or not mask.any():
                        continue
                else:
                    mask = X_test_full[col] == label

                ax.scatter(
                    Y_test_df.loc[mask, param],
                    Y_pred_df.loc[mask, f"{param}_pred"],
                    color=colors[label],
                    label=label.title(),
                    alpha=0.7
                )

                line = [Y_test_df[param].min(), Y_test_df[param].max()]
                ax.plot(line, line, 'k--', linewidth=1)

            leg = ax.legend(
                title=col.title() if col != 'crystal_system' else 'Crystal System',
                prop={'size': 16, 'weight': 'bold'}
            )
            leg.get_title().set_fontsize(18)
            leg.get_title().set_fontweight('bold')

        else:
            ax.scatter(Y_test_df[param], Y_pred_df[f"{param}_pred"], alpha=0.7)
            line = [Y_test_df[param].min(), Y_test_df[param].max()]
            ax.plot(line, line, 'r--', linewidth=1)

        # Bold axis labels, title, ticks
        ax.set_xlabel("Actual Value", fontsize=20, fontweight='bold')
        ax.set_ylabel("Predicted Value", fontsize=20, fontweight='bold')
        ax.set_title(f"Lattice Parameter '{param}' (Å)", fontsize=22, fontweight='bold')
        ax.tick_params(labelsize=18)
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontweight('bold')

        # Display statistics box (bold text inside)
        if displayStatistics and MSEVals is not None and MAEVals is not None and R2Vals is not None:
            stats = (
                f"MSE={MSEVals[i]:.4f}\n"
                f"MAE={MAEVals[i]:.4f}\n"
                f"R²={R2Vals[i]:.4f}"
            )
            if df is not None:
                stats += f"\nFull Dataset Size={len(df)}"
            ax.text(
                0.05, 0.95, stats,
                transform=ax.transAxes, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray'),
                fontsize=16, fontweight='bold'
            )

    fig.suptitle(
        f"Predicted vs. Actual Lattice Parameter Using\n{modelName}",
        fontsize=24, fontweight='bold'
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# Initialize dataset paths (Experimantlly Observed)
# MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized_Experimental.csv"
# TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset_Experimental_TrainTest.csv"
# ModelMetricsDir = "<local_path>/FuelComp/MachineLearning/Dataset/ModelMetrics_Experimental_TrainTest.csv"
# PredictionDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/DatasetPredictions_Experimental_TrainTest.csv"

# # Initialize dataset paths (Theoretical Included)
MPDatasetFeaturized = "<local_path>/FuelComp/MachineLearning/Dataset/MP_Dataset_Featurized_Full.csv"
TrainingDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/Training_Dataset_Full_AvgOnly_TrainTest.csv"
ModelMetricsDir = "<local_path>/FuelComp/MachineLearning/Dataset/ModelMetrics_Full_AvgOnly_TrainTest.csv"
PredictionDatasetDir = "<local_path>/FuelComp/MachineLearning/Dataset/DatasetPredictions_Full_AvgOnly_TrainTest.csv"

In [ ]:
# Read in featurized dataset and feature labels

df = pd.read_csv(MPDatasetFeaturized, low_memory=False)
display(df.head())
display(len(df))

featureLabels = joblib.load('FeatureLabels.joblib')
cs_featureLabels = joblib.load("cs_FeatureLabels.joblib")
sg_featureLabels = joblib.load("sg_FeatureLabels.joblib")

In [ ]:
# Filter out single element materials with identical reduced compositions and spacegroup numbers

df = df.sort_values("formation_energy_per_atom", ascending=True)

nelementsMask = df["nelements"] == 1
dupesMask = df.duplicated(subset=["composition_reduced", "spacegroup_num", "nsites"], keep="first")
mask = ~((nelementsMask) & (dupesMask))

df = df.loc[mask].reset_index(drop=True)

df = df.sort_values("composition_reduced", ascending=True).reset_index(drop=True)
df.groupby('nelements').count()

display(df.head())
display(len(df))

In [ ]:
# Filter out materials with exceptionally large lattice parameters a, b, and c

# Define maximum lattice parameter allowed
latParamThresh = 10 # angstroms

df = df[(df[['a','b','c']] <= latParamThresh).all(axis=1)]

display(df.head())
display(len(df))

In [ ]:
# Filter for materials contiaining a specific crystal system, if desired

filterCrystalSystem = False
csFilter = 'cubic'

if filterCrystalSystem:
    df = df[df['crystal_system'] == csFilter]

    display(df.head())
    display(len(df))

In [ ]:
# Use either space group OHE or crystal system OHE

useSpaceGroupOHE = False

if useSpaceGroupOHE:
    print('Using Spacegrup Number One Hot Encoding!')
    featureLabels = [col for col in featureLabels if col not in cs_featureLabels]
    featureLabels.remove('spacegroup_num')
else:
    print('Using Crystal System One Hot Encoding!')
    featureLabels = [col for col in featureLabels if col not in sg_featureLabels]

print(f'Chosen input features: {featureLabels}')

In [ ]:
# Filter out columns in featureLabels with no values (either all NaN, 0, or False)

emptyColsMask = ((df[featureLabels].isna()) | (df[featureLabels]==0)).all()

emptyCols = pd.Index(featureLabels)[emptyColsMask].tolist()
print(f"Dropped Empty Columns: {emptyCols}")

featureLabels = [col for col in featureLabels if col not in emptyCols]

df = df.drop(columns=emptyCols)

display(df)
print(featureLabels)

In [ ]:
# Specify feature labels to add/remove not originally indluced/removed during original datafeaturization (but feature is present in featurized dataframe)

# Removes any features with these substrings present in the feature label name
toRemove = ['minimum', 'maximum', 'range', 'mode', 'norm', 'max ionic'] # min, max, range, and mode statistics and norms hurt model predictions for impurities

# Adds feature labels with specified name(s)
toAdd = ['nsites']

filtered_featureLabels = [s for s in featureLabels if not any(sub in s for sub in toRemove)]

filtered_featureLabels.extend(toAdd)

print(filtered_featureLabels)

In [ ]:
# Save training dataset and model feature labels

df.to_csv(TrainingDatasetDir, index=False)

joblib.dump(filtered_featureLabels, 'ML_FeatureLabels_Full_AvgOnly.joblib')

In [ ]:
# Load training dataset and model feature labels

df = pd.read_csv(TrainingDatasetDir, low_memory=False)

featureLabels = joblib.load('ML_FeatureLabels_Full_AvgOnly.joblib')

display(df)
print(featureLabels)

In [ ]:
# Create training/testing data for models

Y = df[['a', 'b', 'c']]

X = df[featureLabels]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 57)

mask = X_test['cs_cubic'] == True

print("Dataset Size:", len(df))
print("Cubic Only Size:", len(df[df['crystal_system'] == 'cubic']))
print("Test Size:", len(Y_test))
print("Cubic Only Test Size:", len(Y_test[mask]))

In [ ]:
# Create and train Multi-Output Linear Regression model

linModel = MultiOutputRegressor(LinearRegression())

linModel.fit(X_train, Y_train)

In [ ]:
# Save model

joblib.dump(linModel, 'LinearRegressionModel_Full_TrainTest.joblib')

In [ ]:
# Load saved model

linModel = joblib.load('LinearRegressionModel_Full_TrainTest.joblib')

In [ ]:
# Display statistics for Multi-Output Linear Regression model

# Predict on test dataset
Y_pred_lin = linModel.predict(X_test)

# Print mean squared error, mean absolute error, and R2 values
MSElin = mean_squared_error(Y_test, Y_pred_lin, multioutput='raw_values')
MAElin = mean_absolute_error(Y_test, Y_pred_lin, multioutput='raw_values')
R2lin = r2_score(Y_test, Y_pred_lin, multioutput='raw_values')
print("MSE:", MSElin)
print("MAE:", MAElin)
print("R²:", R2lin)
MSElin_cubic = mean_squared_error(Y_test[mask], Y_pred_lin[mask], multioutput='raw_values')
MAElin_cubic = mean_absolute_error(Y_test[mask], Y_pred_lin[mask], multioutput='raw_values')
R2lin_cubic = r2_score(Y_test[mask], Y_pred_lin[mask], multioutput='raw_values')
print("MSE Cubic Only:", MSElin_cubic)
print("MAE Cubic Only:", MAElin_cubic)
print("R2 Cubic Only:", R2lin_cubic)

# Convert predicted values to dataframe
Y_pred_lin = pd.DataFrame(Y_pred_lin, columns=['a_pred', 'b_pred', 'c_pred'])

# Plot predicted values vs. actual values
PlotPredictions(Y_test, Y_pred_lin, "Multi-Output Linear Regression Model - 80/20 Train/Test Split", True, MSElin_cubic, MAElin_cubic, R2lin_cubic, df, 'crystal_system', ['cubic'], True)

In [ ]:
linModel = []

In [ ]:
# Create and train Lumped Random Forest Regressor model

# This model creates 1 random forest regressor where each tree in the forest predicts lattice parameters a, b, and c simultaneously
rf1Model = RandomForestRegressor(n_estimators=600, random_state=42, n_jobs=-1)

rf1Model.fit(X_train, Y_train)

In [ ]:
# Save model

joblib.dump(rf1Model, 'LumpedRFModel_Full_TrainTest.joblib')

In [ ]:
# Load saved model

rf1Model = joblib.load('LumpedRFModel_Full_TrainTest.joblib')

In [ ]:
# Display statistics for Lumped Random Forest Regressor model

# Predict on test dataset
Y_pred_rf1 = rf1Model.predict(X_test)

# Print mean squared error, mean absolute error, and R2 values
MSErf1 = mean_squared_error(Y_test, Y_pred_rf1, multioutput='raw_values')
MAErf1 = mean_absolute_error(Y_test, Y_pred_rf1, multioutput='raw_values')
R2rf1 = r2_score(Y_test, Y_pred_rf1, multioutput='raw_values')
print("MSE:", MSErf1)
print("MAE:", MAErf1)
print("R²:", R2rf1)
MSErf1_cubic = mean_squared_error(Y_test[mask], Y_pred_rf1[mask], multioutput='raw_values')
MAErf1_cubic = mean_absolute_error(Y_test[mask], Y_pred_rf1[mask], multioutput='raw_values')
R2rf1_cubic = r2_score(Y_test[mask], Y_pred_rf1[mask], multioutput='raw_values')
print("MSE Cubic Only: ", MSErf1_cubic)
print("MAE Cubic Only: ", MAErf1_cubic)
print("R2 Cubic Only: ", R2rf1_cubic)

# Convert predicted values to dataframe
Y_pred_rf1 = pd.DataFrame(Y_pred_rf1, columns=['a_pred', 'b_pred', 'c_pred'])

# Plot predicted values vs. actual values
PlotPredictions(Y_test, Y_pred_rf1, "Lumped RF Regressor Model - 80/20 Train/Test Split", True, MSErf1_cubic, MAErf1_cubic, R2rf1_cubic, df, 'crystal_system', ['cubic'], True)

In [ ]:
rf1Model = []

Model Comparisons:

Full Model (Crystal System OHE)
MSE: [0.53316798 0.5678451  0.63161987]
MAE: [0.39690684 0.40787927 0.44980947]
R²: [0.8277976  0.8144511  0.80086239]
Dataset Size 64785
MSE Cubic Only:  [0.11220001 0.11220001 0.11096802]
MAE Cubic Only:  [0.14008169 0.14008169 0.13923083]
R2 Cubic Only:  [0.96077612 0.96077612 0.96120681]
Cubic Only Test Size: 3187

Cubic Only (SG OHE)
MSE: [0.16597781 0.16597781 0.16597781]
MAE: [0.15834165 0.15834165 0.15834165]
R²: [0.94424109 0.94424109 0.94424109]
Dataset Size: 15935
Cubic Only Test Size: 3187

Cubic Only (No OHE)
MSE: [0.1561258 0.1561258 0.1561258]
MAE: [0.1535477 0.1535477 0.1535477]
R²: [0.94755079 0.94755079 0.94755079]
Dataset Size: 15935
Cubic Only Test Size: 3187

In [ ]:
# Create and train Independent Random Forest Regressor model

# This model creates 3 independent random forest regressors for each lattice parameters a, b, and c
rf2Model = MultiOutputRegressor(RandomForestRegressor(n_estimators=600, random_state=42, n_jobs=-1))

rf2Model.fit(X_train, Y_train)

In [ ]:
# Save model

joblib.dump(rf2Model, 'IndependentRFModel_Full_TrainTest.joblib')

In [ ]:
# Load saved model

rf2Model = joblib.load('IndependentRFModel_Full_TrainTest.joblib')

In [ ]:
# Display statistics for Independent Random Forest Regressor model

# Predict on test dataset
Y_pred_rf2 = rf2Model.predict(X_test)

# Print mean squared error, mean absolute error, and R2 values
MSErf2 = mean_squared_error(Y_test, Y_pred_rf2, multioutput='raw_values')
MAErf2 = mean_absolute_error(Y_test, Y_pred_rf2, multioutput='raw_values')
R2rf2 = r2_score(Y_test, Y_pred_rf2, multioutput='raw_values')
print("MSE:", MSErf2)
print("MAE:", MAErf2)
print("R²:", R2rf2)
MSErf2_cubic = mean_squared_error(Y_test[mask], Y_pred_rf2[mask], multioutput='raw_values')
MAErf2_cubic = mean_absolute_error(Y_test[mask], Y_pred_rf2[mask], multioutput='raw_values')
R2rf2_cubic = r2_score(Y_test[mask], Y_pred_rf2[mask], multioutput='raw_values')
print("MSE Cubic Only: ", MSErf2_cubic)
print("MAE Cubic Only: ", MAErf2_cubic)
print("R2 Cubic Only: ", R2rf2_cubic)

# Convert predicted values to dataframe
Y_pred_rf2 = pd.DataFrame(Y_pred_rf2, columns=['a_pred', 'b_pred', 'c_pred'])

# Plot predicted values vs. actual values
PlotPredictions(Y_test, Y_pred_rf2, "Independent RF Regressor Model - 80/20 Train/Test Split", True, MSErf2_cubic, MAErf2_cubic, R2rf2_cubic, df, 'crystal_system', ['cubic'], True)

In [ ]:
rf2Model = []

In [ ]:
# Create and train XGBoost Gradient Boosting Regressor model

# This model natively predicts lattice parameters a, b, and c simultaneously
gbr1Model = XGBRegressor(n_estimators=1800, learning_rate=0.05, max_depth=10, subsample=0.8, tree_method='hist', multi_strategy='multi_output_tree', random_state=42, n_jobs=-1)

gbr1Model.fit(X_train, Y_train)

In [ ]:
# Save model

joblib.dump(gbr1Model, 'XGBoostGBRModel_Full_TrainTest.joblib')

In [ ]:
# Load saved model

gbr1Model = joblib.load('XGBoostGBRModel_Full_TrainTest.joblib')

In [ ]:
# Display statistics for Scikit-Learn Gradient Boosting Regressor

# Predict on test dataset
Y_pred_gbr1 = gbr1Model.predict(X_test)

# Print mean squared error, mean absolute error, and R2 values
MSEgbr1 = mean_squared_error(Y_test, Y_pred_gbr1, multioutput='raw_values')
MAEgbr1 = mean_absolute_error(Y_test, Y_pred_gbr1, multioutput='raw_values')
R2gbr1 = r2_score(Y_test, Y_pred_gbr1, multioutput='raw_values')
print("MSE:", MSEgbr1)
print("MAE:", MAEgbr1)
print("R²:", R2gbr1)
MSEgbr1_cubic = mean_squared_error(Y_test[mask], Y_pred_gbr1[mask], multioutput='raw_values')
MAEgbr1_cubic = mean_absolute_error(Y_test[mask], Y_pred_gbr1[mask], multioutput='raw_values')
R2gbr1_cubic = r2_score(Y_test[mask], Y_pred_gbr1[mask], multioutput='raw_values')
print("MSE Cubic Only: ", MSEgbr1_cubic)
print("MAE Cubic Only: ", MAEgbr1_cubic)
print("R2 Cubic Only: ", R2gbr1_cubic)

# Convert predicted values to dataframe
Y_pred_gbr1 = pd.DataFrame(Y_pred_gbr1, columns=['a_pred', 'b_pred', 'c_pred'])

# Plot predicted values vs. actual values
PlotPredictions(Y_test, Y_pred_gbr1, "Lumped GBR Model - 80/20 Train/Test Split", True, MSEgbr1_cubic, MAEgbr1_cubic, R2gbr1_cubic, df, 'crystal_system', ['cubic'], True)

In [ ]:
gbr1Model = []

In [ ]:
# Create and train Scikit-Learn Gradient Boosting Regressor model

# This model creates 3 independent gradient boosting regressors for each lattice parameters a, b, and c
gbr2Model = MultiOutputRegressor(HistGradientBoostingRegressor(max_iter=1500, learning_rate=0.05, max_depth=10, max_features=0.8, random_state=42))

gbr2Model.fit(X_train, Y_train)

In [ ]:
# Save model

joblib.dump(gbr2Model, 'ScikitLearnGBRModel_Full_TrainTest.joblib')

In [ ]:
# Load saved model

gbr2Model = joblib.load('ScikitLearnGBRModel_Full_TrainTest.joblib')

In [ ]:
# Display statistics for Scikit-Learn Gradient Boosting Regressor

# Predict on test dataset
Y_pred_gbr2 = gbr2Model.predict(X_test)

# Print mean squared error, mean absolute error, and R2 values
MSEgbr2 = mean_squared_error(Y_test, Y_pred_gbr2, multioutput='raw_values')
MAEgbr2 = mean_absolute_error(Y_test, Y_pred_gbr2, multioutput='raw_values')
R2gbr2 = r2_score(Y_test, Y_pred_gbr2, multioutput='raw_values')
print("MSE:", MSEgbr2)
print("MAE:", MAEgbr2)
print("R²:", R2gbr2)
MSEgbr2_cubic = mean_squared_error(Y_test[mask], Y_pred_gbr2[mask], multioutput='raw_values')
MAEgbr2_cubic = mean_absolute_error(Y_test[mask], Y_pred_gbr2[mask], multioutput='raw_values')
R2gbr2_cubic = r2_score(Y_test[mask], Y_pred_gbr2[mask], multioutput='raw_values')
print("MSE Cubic Only: ", MSEgbr2_cubic)
print("MAE Cubic Only: ", MAEgbr2_cubic)
print("R2 Cubic Only: ", R2gbr2_cubic)

# Convert predicted values to dataframe
Y_pred_gbr2 = pd.DataFrame(Y_pred_gbr2, columns=['a_pred', 'b_pred', 'c_pred'])

# Plot predicted values vs. actual values
PlotPredictions(Y_test, Y_pred_gbr2, "Independent GBR Model - 80/20 Train/Test Split", True, MSEgbr2_cubic, MAEgbr2_cubic, R2gbr2_cubic, df, 'crystal_system', ['cubic'], True)

In [ ]:
gbr2Model = []

In [ ]:
# Create and save ML metrics dataframe

metrics_df = pd.DataFrame(['Linear Regression', 'Lumped RF', 'Independent RF', 'Lumped GBR', 'Independent GBR'], columns=['model_name'])

metrics = [
    'MSE', 'MAE', 'R2',
    'MSE_cubic', 'MAE_cubic', 'R2_cubic'
]
params = ['a', 'b', 'c']

for m in metrics:
    for p in params:
        metrics_df[f"{m}_{p}"] = None

metrics_dict = {
    "Linear Regression": (MSElin, MAElin, R2lin, MSElin_cubic, MAElin_cubic, R2lin_cubic),
    "Lumped RF": (MSErf1, MAErf1, R2rf1, MSErf1_cubic, MAErf1_cubic, R2rf1_cubic),
    "Independent RF": (MSErf2, MAErf2, R2rf2, MSErf2_cubic, MAErf2_cubic, R2rf2_cubic),
    "Lumped GBR": (MSEgbr1, MAEgbr1, R2gbr1, MSEgbr1_cubic, MAEgbr1_cubic, R2gbr1_cubic),
    "Independent GBR": (MSEgbr2, MAEgbr2, R2gbr2, MSEgbr2_cubic, MAEgbr2_cubic, R2gbr2_cubic)
}

for idx, row in metrics_df.iterrows():
    name = row['model_name']
    MSE, MAE, R2, MSE_cubic, MAE_cubic, R2_cubic = metrics_dict[name]
    metrics_df.loc[idx, ["MSE_a", "MSE_b", "MSE_c"]] = MSE
    metrics_df.loc[idx, ["MAE_a", "MAE_b", "MAE_c"]] = MAE
    metrics_df.loc[idx, ["R2_a", "R2_b", "R2_c"]] = R2
    metrics_df.loc[idx, ["MSE_cubic_a", "MSE_cubic_b", "MSE_cubic_c"]] = MSE_cubic
    metrics_df.loc[idx, ["MAE_cubic_a", "MAE_cubic_b", "MAE_cubic_c"]] = MAE_cubic
    metrics_df.loc[idx, ["R2_cubic_a", "R2_cubic_b", "R2_cubic_c"]] = R2_cubic

display(metrics_df)

metrics_df.to_csv(ModelMetricsDir, index=False)

In [ ]:
# Append predictions from each model to training dataframe and save predicted dataframe

Y_pred_lin = Y_pred_lin.add_suffix('_lin')
Y_pred_rf1 = Y_pred_rf1.add_suffix('_rf1')
Y_pred_rf2 = Y_pred_rf2.add_suffix('_rf2')
Y_pred_gbr1 = Y_pred_gbr1.add_suffix('_gbr1')
Y_pred_gbr2 = Y_pred_gbr2.add_suffix('_gbr2')

df = pd.concat([df, Y_pred_lin, Y_pred_rf1, Y_pred_rf2, Y_pred_gbr1, Y_pred_gbr2], axis=1)

display(df)

df.to_csv(PredictionDatasetDir, index=False)